# Canada's Health Profile: Exploratory Data Analysis of WHO Indicators

Maestría en Ciencias Computacionales. Series de Tiempo
Ana Sofía Rentería Palomares
Class Project U1–U2: Executive EDA Report

**Source:** World Health Organization, *Health data overview for Canada*: https://data.who.int/countries/124



### Libraries

In [ ]:
# Hide warning messages 
import warnings
warnings.filterwarnings("ignore")
# Make the Nixtla libraries return unique_id as a column instead of as the index
import os
os.environ["NIXTLA_ID_AS_COL"] = "true"
# numpy: numerical arrays. suppress=True prints numbers without scientific notation
import numpy as np
np.set_printoptions(suppress=True)
np.random.seed(1)   # fixed seed so the results are reproducible
import random
random.seed(1)      # same idea, for Python's own random module
# pandas: tables (DataFrames). Show up to 100 characters per column and 3 decimals
import pandas as pd
pd.set_option("max_colwidth", 100)
pd.set_option("display.precision", 3)
# plot_series from utilsforecast, renamed so it does not clash with fpppy's plot_series below
from utilsforecast.plotting import plot_series as plot_series_utils
# seaborn and matplotlib: static plots and their style
import seaborn as sns
sns.set_style("whitegrid")
import matplotlib.pyplot as plt
plt.style.use("ggplot")
# Default size, resolution and font sizes for every matplotlib figure
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "figure.constrained_layout.use": True,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "legend.title_fontsize": 10,
    "grid.alpha": 1.0,
})
# Default line colours for matplotlib (black)
import matplotlib as mpl
from cycler import cycler
mpl.rcParams['axes.prop_cycle'] = cycler(color=["#000000", "#000000"])
# plot_series from fpppy
from fpppy.utils import plot_series

# tsfeatures: computes summary statistics / features of a time series
import tsfeatures as tsf

In [ ]:
# Interactive plots
from pathlib import Path                      # handles folder and file paths
import plotly.graph_objects as go             # go.Figure: build a plotly figure trace by trace
import plotly.io as pio                       # plotly settings (renderer, template)
from plotly.subplots import make_subplots     # grids with several panels in one figure

pio.renderers.default = "plotly_mimetype+notebook_connected"   # how figures are displayed in the notebook
pio.templates.default = "plotly_white"                         # white background for every figure

# FIG: folder where the PNG copies of the figures are saved (created if it does not exist)
FIG = Path("figures/population")
FIG.mkdir(parents=True, exist_ok=True)

# show(fig, name): fig = plotly figure, name = file name of the PNG (without extension),
# width / height = size of the figure in pixels
def show(fig, name, width=1000, height=450):
    # Shows the interactive figure and saves a PNG copy for the report
    fig.update_layout(width=width, height=height, title_x=0.01, margin=dict(t=70, l=60, r=30, b=50))
    fig.write_image(FIG / f"{name}.png", scale=2)
    fig.show()

## Part 1: Population

Dataset: `population.xlsx` : total population of Canada by single age (0, 1, …, 99, 100+), both sexes,
in thousands, annual estimates 1950–2023.

### 1. Loading the data

In [ ]:
# raw: the Excel file as downloaded; header=1 means the column names are in the second row
raw = pd.read_excel("population.xlsx", header=1)
raw = raw[raw["Region, subregion, country or area *"] == "Canada"] # filters the table to keep only the rows for Canada
raw.iloc[:3, :16]   # preview: first 3 rows and first 16 columns

**Explanation:** **We compute the two series.**
* `pop_age`: year-by-age table (74 years × 101 ages).
* **Total population** = sum of all ages for each year (`sum(axis=1)`), in millions.
* **Annual growth rate** = how much the population grew relative to the previous year, in %. It is computed with `pct_change()`:
  (this year's population − previous year's population) / previous year's population × 100.
  **1950 is empty** (NaN) because it has no previous year. **This is the series we analyze.**

In [ ]:
# pop_age: year x age table. The year becomes the index and columns 11 to 111 are the ages (0, 1, ..., 100+), in thousands
pop_age = raw.set_index(raw["Year"].astype(int))[raw.columns[11:112]].astype(float)
pop_age.index.name = "year"
# total: total population of each year = sum of all ages (axis=1 sums across the columns), in thousands
total = pop_age.sum(axis=1)

# pop: table with the two series, one row per year
pop = pd.DataFrame({
    "Total population (millions)": total / 1000,           # thousands -> millions
    "Annual growth rate (%)": total.pct_change() * 100,    # % change vs. the previous year (1950 is NaN)
})
# Check the range of years and that there are no missing values
print("Years:", pop.index.min(), "-", pop.index.max(), "missing values:", pop_age.isna().sum().sum())
pop.loc[[1950, 1951, 1970, 1990, 2010, 2020, 2021, 2023]]   # show a few selected years

**Explanation:** **The series we are going to analyze (`growth`).**
* `growth` = the growth rate without the empty 1950 value (`dropna()`): 73 values, from 1951 to 2023.
* `growth_long` = the same series in long format (`unique_id`, `ds`, `y`), which `tsfeatures` requires.

In [ ]:
# growth: the annual growth rate without the empty 1950 value (index = year, values = %)
growth = pop["Annual growth rate (%)"].dropna()
# growth_long: the same series in long format (unique_id, ds, y)
growth_long = pd.DataFrame({"unique_id": "Annual growth rate (%)",                    # name of the series, repeated on every row
                            "ds": pd.to_datetime(growth.index.astype(str)),           # converts the years to text and then to real dates
                            "y": growth.values})                                      # the value of the growth rate
growth_long.head()   # first 5 rows

### 2. Summary statistics (`tsfeatures`)


* `freq=1` because the data are **annual**.
* `threads=1` keeps it from hanging on Mac.

The rate ranges from a minimum of 0.74 % to a maximum of 2.98 %; half of the years grew between approximately 1.05 % and approximately 1.77 % (p25–p75).

In [ ]:
# summary_stats: summary statistics of the series computed with tsfeatures
# freq=1 -> annual data, features=[tsf.statistics] -> min, percentiles, median, max, mean, ...
# scale=False -> keep the original units (%), threads=1 -> one process (avoids hanging on Mac)
summary_stats = tsf.tsfeatures(growth_long, freq=1, features=[tsf.statistics], scale=False, threads=1)
summary_stats[["unique_id", "min", "p25", "median", "p75", "max", "mean"]]   # keep only the columns we report

- Minimum (min = 0.739): the year with the lowest growth; the population grew by 0.739 %.
- 25th percentile (p25 = 1.046): in one quarter of the years, growth was 1.046 % or less.
- Median (median = 1.224): half of the years grew by less than 1.224 % and the other half by more.
- 75th percentile (p75 = 1.771): in three quarters of the years, growth was 1.771 % or less.
- Maximum (max = 2.981): the year with the highest growth, 2.981 %.
- Mean (mean = 1.451): the average across all years.


### 3. Time plots

In [ ]:
# pop_long: both series (total population and growth rate) stacked in long format (unique_id, ds, y)
pop_long = (pop.reset_index()                                                    # the year goes back to being a column
            .melt(id_vars="year", var_name="unique_id", value_name="y")          # wide -> long: one row per year and series
            .dropna()                                                            # removes the empty 1950 growth rate
            .assign(ds=lambda x: pd.to_datetime(x["year"].astype(str)))          # year -> real date
            [["unique_id", "ds", "y"]])

# One panel per series (max_ids=2), in the order of the table (plot_random=False)
fig = plot_series_utils(pop_long, max_ids=2, plot_random=False, engine="plotly")
# axis labels: replace the default "Time [ds]" / "Target [y]" of plot_series
labels = {"Time [ds]": "Year", "Target [y]": ""}
fig.for_each_annotation(lambda a: a.update(text=labels.get(a.text, a.text.replace("unique_id=", ""))))
fig.update_yaxes(title_text="Population (millions)", row=1, col=1)
fig.update_yaxes(title_text="Growth rate (% vs. previous year)", row=1, col=2)
fig.update_layout(title="Canada – total population and annual growth rate, 1950–2023", showlegend=False)
show(fig, "01_time_plots", height=650)

### 4. Lag plots: annual growth rate



In [ ]:
# lag_plot: 3x3 grid of lag plots, each one compares y(t) with y(t-k)
#   s = series indexed by year, title = title of the figure, name = file name of the PNG,
#   lags = which lags to draw (1 to 9 by default)
def lag_plot(s, title, name, lags=range(1, 10)):
    fig = make_subplots(rows=3, cols=3, subplot_titles=[f"lag {k}" for k in lags])
    lo, hi = s.min(), s.max()   # lowest and highest value of the series: the two ends of the diagonal
    for i, k in enumerate(lags):
        row, col = i // 3 + 1, i % 3 + 1   # position of this panel in the grid
        # df: each year's value (y) next to the value k years earlier (lag); shift(k) moves the series k years
        df = pd.DataFrame({"y": s, "lag": s.shift(k)}).dropna()
        # points: x = value k years earlier, y = current value, colour = year (colour bar only on the first panel)
        fig.add_scatter(x=df["lag"], y=df["y"], mode="markers", row=row, col=col,
                        marker=dict(size=7, color=df.index, colorscale="Viridis", cmin=s.index.min(), cmax=s.index.max(),
                                    showscale=(i == 0), colorbar=dict(title="Year")),
                        customdata=df.index, hovertemplate="year %{customdata}<br>y(t-k) = %{x:.2f}<br>y(t) = %{y:.2f}<extra></extra>")
        # dashed diagonal y = x: points on it have the same value as k years earlier
        fig.add_scatter(x=[lo, hi], y=[lo, hi], mode="lines", line=dict(color="gray", dash="dash", width=1),
                        row=row, col=col, hoverinfo="skip")
    fig.update_layout(title=title, showlegend=False)
    show(fig, name, width=950, height=850)

**Explanation:** **What the lag plots of the growth rate show:**
* **The purple points (1950s)** sit together in the upper right: in that period the rate was high (approximately 2.5–3 %) and one year
  looked like the next.
* **The 1960s–70s** (dark blue), at the large lags (6–9), fall **below** the diagonal: for example, approximately 1.9 % in one year
  versus approximately 2.8 % nine years earlier. Each year was lower than years before: **negative trend**.
* **The recent years** (green/yellow) are clustered near 1 %, around the diagonal: the rate has stabilized.
* **Lag 1:** the points follow the diagonal fairly closely (one year resembles the next). **As the lag increases** the cloud
  spreads out: the resemblance fades. Since 1970, the correlation drops from 0.81 (lag 1) to 0.47 (lag 2) and to almost 0 at lags 7–9.
* **No lag stands out** from the others: **there is no seasonality**.

In [ ]:
# Lag plots of the annual growth rate (lags 1 to 9)
lag_plot(growth, "Lag plots: annual growth rate (%)", "02_lag_growth")

**Interpretation of the lag plot (growth rate):**
* Looking at lag 1 we can see that **there is temporal dependence**, because the points are very close to the diagonal. This means that the growth rate of one year is very similar to that of the previous year, so there is **positive autocorrelation**. At **lags 2 and 3** the values still resemble each other. However, the cloud starts to open up, which means the resemblance fades as we look further back in time.
* **The current rate is lower than it used to be.** At **lags 6 to 9** most of the points fall **below** the diagonal. The ones furthest away are those of the 1960s. A **negative trend** can be seen.
* The **purple** points of the 1950s sit together in the upper right, showing that it was the period of highest growth (2.4–3 %).
  The **green and yellow** points (since 2000) are clustered in the lower left, near 1 % and on the diagonal; here we can see that the rate **no longer changes much**: it stabilized at close to 1 %.
* **Outliers.** The observations for 1970–1971 fall **above** the diagonal at the first lags (the rate jumped from 1.46 % to 2.13 %) and 2021 falls **below**, with a value of 0.74 %, the minimum of the series, most likely caused by the COVID pandemic.
* **No lag stands out** from the others: each one resembles the previous one, so we can say that **there is no seasonality**.

**Conclusion:** this lag plot confirms that Canada's population
**keeps growing, but more and more slowly**: from almost 3 % per year in the 1950s to close to 1 % since the 1990s. As a result, the distant past predicts the present less and less well. This reflects factors such as the decline in the birth rate and the 2021 drop when borders were closed because of COVID-19.

## Part 2: Life expectancy

In [ ]:
# lx: life expectancy at birth in Canada, by sex (FEMALE, MALE, TOTAL)
lx = pd.read_csv("Life expectancy at birth.csv")
lx = lx.loc[lx["GEO_NAME_SHORT"] == "Canada"]                          # keep only the rows for Canada
lx = lx[["GEO_NAME_SHORT", "DIM_TIME", "DIM_SEX", "AMOUNT_N"]]         # keep only the columns we need
lx = lx.rename(columns={"GEO_NAME_SHORT": "country", "DIM_TIME": "year", "DIM_SEX": "sex", "AMOUNT_N": "life_expectancy"})
lx = lx.sort_values(["sex", "year"])                                   # sort by sex and then by year
lx = lx.set_index("year")                                              # the year becomes the index
lx

**Explanation:** **Time plot of life expectancy by sex.**
* The `for` loop goes through `FEMALE`, `TOTAL` and `MALE`, filters the rows of each one and draws a line (orange = women, black = both
  sexes, blue = men). `d.index` goes on the X axis because the year is the index.
* `hovermode="x unified"`: hovering over a year shows the 3 values together.

**What it shows:** the 3 lines **rise** from 2000 to 2019 (**positive trend**), **fall in 2020** (COVID-19) and recover slightly
in 2021. Women always live longer than men.

In [ ]:
# colors / names: line colour and legend label for each sex
colors = {"FEMALE": "#eb6834", "MALE": "#2a78d6", "TOTAL": "black"}
names = {"FEMALE": "Female", "MALE": "Male", "TOTAL": "Both sexes"}

fig = go.Figure()
# one line per sex
for sex in ["FEMALE", "TOTAL", "MALE"]:
    d = lx.loc[lx["sex"] == sex]   # d: the rows of this sex
    fig.add_scatter(x=d.index, y=d["life_expectancy"], mode="lines+markers",
                    name=names[sex], line=dict(color=colors[sex], width=2.5))
# hovermode="x unified": hovering over a year shows the 3 values together
fig.update_layout(title="Life expectancy at birth, Canada 2000–2021",
                  xaxis_title="Year", yaxis_title="Years", hovermode="x unified")
show(fig, "09_life_expectancy_by_sex")

**Interpretation:** the 3 lines **rise** from 2000 to 2019 (**positive trend**), **fall in 2020** (COVID-19) and recover slightly
in 2021. Women always live longer than men.

**Explanation:** **Function for the lag plots** (same as `lag_plot` from Part 1).
* `xl` is the table with each year's value (`y`) and the value from *k* years earlier (`lag`).


In [ ]:
# lag_plot_lx: same as lag_plot from Part 1 (the hover text shows 1 decimal instead of 2)
#   s = series indexed by year, title = title of the figure, name = file name of the PNG, lags = which lags to draw
def lag_plot_lx(s, title, name, lags=range(1, 10)):
    fig = make_subplots(rows=3, cols=3, subplot_titles=[f"lag {k}" for k in lags])
    lo, hi = s.min(), s.max()   # the two ends of the diagonal
    for i, k in enumerate(lags):
        row, col = i // 3 + 1, i % 3 + 1   # position of this panel in the grid
        # xl: each year's value (y) next to the value k years earlier (lag)
        xl = pd.DataFrame({"y": s, "lag": s.shift(k)}).dropna()
        # points coloured by year
        fig.add_scatter(x=xl["lag"], y=xl["y"], mode="markers", row=row, col=col,
                        marker=dict(size=7, color=xl.index, colorscale="Viridis", cmin=s.index.min(), cmax=s.index.max(),
                                    showscale=(i == 0), colorbar=dict(title="Year")),
                        customdata=xl.index, hovertemplate="year %{customdata}<br>y(t-k) = %{x:.1f}<br>y(t) = %{y:.1f}<extra></extra>")
        # dashed diagonal y = x
        fig.add_scatter(x=[lo, hi], y=[lo, hi], mode="lines", line=dict(color="gray", dash="dash", width=1),
                        row=row, col=col, hoverinfo="skip")
    fig.update_layout(title=title, showlegend=False)
    show(fig, name, width=950, height=850)

**Explanation:** **Lag plots of life expectancy (`TOTAL`, both sexes).** The data are annual: **1 lag = 1 year**.
* `le` = the `TOTAL` rows, `life_expectancy` column: 22 values, one per year (2000–2021).

**What it shows:** at every lag the points form a rising line, **above** the diagonal (each year life expectancy is
higher than *k* years earlier): **positive trend**. The lags resemble each other and **none stands out**: **there is no seasonality**.
The yellow points (2020–2021) sit lower than the rest: that is the drop caused by COVID-19.
In addition, since approximately 2011 (green points) the cloud **flattens**: life expectancy kept rising, but much more slowly
(from approximately 81.4 years in 2011 to approximately 81.7 in 2018).

In [ ]:
# le: life expectancy for both sexes (TOTAL), one value per year (2000–2021)
le = lx.loc[lx["sex"] == "TOTAL", "life_expectancy"]
lag_plot_lx(le, "Lag plots: life expectancy at birth (years), both sexes", "10_lag_life_expectancy")

**Conclusion of the lag plot (life expectancy):**
* At **lags 1, 2, 3 and to some extent 4** the points come very close to the diagonal. This shows temporal dependence and positive autocorrelation. As the lags increase, we can see that the distant past resembles the current year less and less.
* **Current life expectancy is higher than in previous years.** At **lags 6, 7 and 8** the cloud is already
clearly **above** the diagonal, and the three panels resemble each other. We can therefore see a **positive trend**.
* The **green** points (2012–2018) form an almost horizontal band, which means that in those years the value barely changed (from 81.5 to 81.7 years). Life expectancy **stops rising as it approaches 82 years**. The **yellow** points at the end (2020 and 2021) fall **below** the diagonal at every lag: in 2020 it dropped 0.72 years (from 82.0 to 81.3), the only major fall in the series, because of COVID-19.
* **No lag stands out** from the others: each one resembles the previous one, so we can deduce that **there is no seasonality and there are no cycles**.

**Conclusion:** this lag plot confirms that life expectancy in Canada **has been increasing**: this reflects technological factors, health research, medicines and public policy that keep improving the population's quality of life. It also shows that it has now **stabilized** as it approaches 82 years, and that an event such as the COVID-19 pandemic can suddenly affect life expectancy for that year.

## Part 3: Leading causes of death

In [ ]:
# ld: causes of death in Canada with their rate (deaths per 100,000 population)
ld = pd.read_csv("Top causes of death.csv")
ld = ld.loc[(ld["DIM_COUNTRY_CODE"] == "CAN")]                         # keep only the rows for Canada
ld = ld[["DIM_GHECAUSE_TITLE", "VAL_DTHS_RATE100K_NUMERIC"]]           # keep the cause and its rate
ld = ld.rename(columns={"DIM_GHECAUSE_TITLE": "Cause of death", "VAL_DTHS_RATE100K_NUMERIC": "Deaths_per_100000"})
ld = ld.set_index("Cause of death")                                    # the cause becomes the index
ld

In [ ]:
ld.index.nunique() # how many different causes of death there are

In [ ]:
# top10: the 10 causes with the highest death rate
top10 = ld.sort_values("Deaths_per_100000", ascending=False).head(10)
top10 = top10.sort_values("Deaths_per_100000")      # from lowest to highest, so the highest ends up at the top

# horizontal bar chart, with the value written next to each bar
fig = go.Figure()
fig.add_bar(x=top10["Deaths_per_100000"], y=top10.index, orientation="h",
            marker_color="#4db8df", text=top10["Deaths_per_100000"].round(1), textposition="outside")
fig.update_layout(title="Top 10 causes of death, Canada 2021",
                  xaxis_title="Deaths per 100,000 population", yaxis_title="")
show(fig, "24_top10_causes_of_death", height=500)


## Part 4: Health Statistics

Risk factors and environment: hypertension, alcohol consumption, adult obesity, safe drinking water and tobacco use
(WHO, Canada, both sexes).


In [ ]:
# time_plot: line chart of one series over time
#   s = series indexed by year, title = title of the figure, ytitle = label of the Y axis,
#   name = file name of the PNG, color = colour of the line
def time_plot(s, title, ytitle, name, color="#86d9f2"):
    fig = go.Figure()
    fig.add_scatter(x=s.index, y=s, mode="lines+markers", line=dict(color=color, width=2.5))
    fig.update_layout(title=title, xaxis_title="Year", yaxis_title=ytitle, showlegend=False)
    show(fig, name)

### 4.1 Hypertension (adults aged 30–79, %)

**Explanation:** **Hypertension data.** We filter Canada and both sexes (`TOTAL`), keep only the year and the value, rename the columns, sort by year and set the year as the index. This leaves 30 years (1990–2019) with a single column: the **% of adults aged
30 to 79 with hypertension**.

In [ ]:
# hy: % of adults aged 30–79 with hypertension, one row per year
hy = pd.read_csv("Hypertension.csv")
hy = hy.loc[(hy["GEO_NAME_SHORT"] == "Canada") & (hy["DIM_SEX"] == "TOTAL")]   # Canada, both sexes
hy = hy[["DIM_TIME", "RATE_PER_100_N"]]                                        # keep the year and the value
hy = hy.rename(columns={"DIM_TIME": "year", "RATE_PER_100_N": "hypertension_rate"})
hy = hy.sort_values("year").set_index("year")                                  # sort by year and set it as the index
hy

**Explanation:** **Time plot** of hypertension.

In [ ]:
# Time plot of hypertension
time_plot(hy["hypertension_rate"], "Hypertension in adults (age 30–79), Canada 1990–2019",
          "% of adults aged 30–79", "11_hypertension")

**Interpretation:** the prevalence of hypertension among adults aged 30 to 79 has fallen considerably and progressively, from 33 % in 1990 to 22.1 % in 2019, showing a negative trend. It falls slightly faster in the 1990s than in the 2010s.

In [ ]:
# Lag plots of hypertension (lags 1 to 9)
lag_plot(hy["hypertension_rate"], "Lag plots: hypertension in adults (%)", "12_lag_hypertension")

**Interpretation of the lag plot (hypertension):**
* At **lag 1** the points form a straight line right next to the diagonal, which means that hypertension in one year is almost the same as in the previous year. There is therefore a very strong **positive autocorrelation**: knowing the value of one year, the next one can be known almost exactly.
* It can also be seen that **current hypertension is lower than in previous years.** At **every** lag, all the points are **below** the diagonal, and the line moves further and further away. We can therefore say that there is a steady **negative trend**.
* The **purple** points (1990s) are the furthest from the diagonal and the **yellow** ones (2015–2019) the
closest: the prevalence of hypertension keeps falling, but **more slowly**.
* The 9 panels look the same and none stands out, so we can say that **there is no seasonality and there are no cycles**. Since no points fall off the line, we conclude that **there are no atypical years**.

 **Conclusion:** this lag plot confirms that hypertension among adults in Canada **has fallen every year without exception** for 30 years (from 33.0 % to 22.1 %). This reflects possible improvements in detection, in access to medicines and in prevention programmes. The line is so perfect because the data are
 **estimates from a WHO model**, not direct measurements for each year; that is why there is almost no residual.

### 4.2 Alcohol consumption (litres of pure alcohol per person aged 15+)

**Explanation:** **Alcohol consumption data**, with the same procedure as for hypertension. We filter Canada **and** `TOTAL`
and keep only the year and the value. This leaves 23 years (2000–2022). The value is
**litres of pure alcohol per person aged 15 or older, per year**.

In [ ]:
# al: litres of pure alcohol per person aged 15+ per year, one row per year
al = pd.read_csv("Alcohol consumption.csv")
al = al.loc[(al["GEO_NAME_SHORT"] == "Canada") & (al["DIM_SEX"] == "TOTAL")]   # Canada, both sexes
al = al[["DIM_TIME", "RATE_PER_CAPITA_N"]]                                     # keep the year and the value
al = al.rename(columns={"DIM_TIME": "year", "RATE_PER_CAPITA_N": "alcohol_litres"})
al = al.sort_values("year").set_index("year")                                  # sort by year and set it as the index
al

**Explanation:** **Time plot** of alcohol consumption.

In [ ]:
# Time plot of alcohol consumption
time_plot(al["alcohol_litres"], "Alcohol consumption, Canada 2000–2022", "Litres of pure alcohol per capita (15+)",
          "14_alcohol")

 **Interpretation:** Consumption **rises** from 9.4 L (2000) to its maximum of **10.3 L in 2008**, then **falls little by little** to 9.97 L in 2014 and since then stays **stable at around 10 L**. We can therefore deduce that it is not a fixed trend: over those 20 years the direction of the time series changed, from a positive trend until 2008 to a decrease in alcohol consumption that then stabilized at around 10 litres.


In [ ]:
# Lag plots of alcohol consumption (lags 1 to 9)
lag_plot(al["alcohol_litres"], "Lag plots: alcohol consumption (litres per capita)", "15_lag_alcohol")

**Interpretation (alcohol consumption):**
* At **lag 1** the points follow the diagonal, which means that consumption in one year is very similar to that of the previous year. In this panel, as in the other lags, we can see that consumption reached its highest point around 2008–2010 and then started to decrease. At **lags 2 and 3** the points of the first years move away from the diagonal, upwards, because more growth accumulates over two or three years. At these first three lags there is a positive correlation between one year's consumption and that of previous years. However, at **lag 4** the cloud becomes almost horizontal, so consumption four years ago hardly helps to predict the current value. From **lag 5** onwards, and more clearly at **lags 6, 7, 8 and 9**, the points form a line that **slopes downward**.

**Conclusion:** this lag plot confirms that alcohol consumption **does not have a trend in a single direction**: it rose in the 2000s, reached its maximum in 2008 (10.3 L) and since 2014 has started to stabilize.
That is why the recent past (1–2 years) does help to know the current value, but the distant past does not. No improvement is visible: consumption stopped rising, but it did not return to the 2000 level (9.4 L). With a single rise and fall in 23 years we cannot claim that it is a cycle or that there is seasonality.

### 4.3 Adult obesity (adults aged 18+, %)

In [ ]:
# ob: % of adults aged 18+ with obesity, one row per year
ob = pd.read_csv("Adult obesity.csv")
ob = ob.loc[(ob["GEO_NAME_SHORT"] == "Canada") & (ob["DIM_SEX"] == "TOTAL")]   # Canada, both sexes
ob = ob[["DIM_TIME", "RATE_PER_100_N"]]                                        # keep the year and the value
ob = ob.rename(columns={"DIM_TIME": "year", "RATE_PER_100_N": "obesity_rate"})
ob = ob.sort_values("year").set_index("year")                                  # sort by year and set it as the index
ob

In [ ]:
# Time plot of adult obesity
time_plot(ob["obesity_rate"], "Obesity in adults (age 18+), Canada 1990–2022", "% of adults", "17_obesity")

**Interpretation:** Obesity **rises the whole time**: from **13.96 % (1990) to 26.2 % (2022)**, almost **double**. In the 1990s it rose quickly. In the 2000s the pace of the increase slowed down, and since approximately 2010 it rises **more slowly**, but it **never falls**, showing a positive trend.

In [ ]:
# Lag plots of adult obesity (lags 1 to 9)
lag_plot(ob["obesity_rate"], "Lag plots: obesity in adults (%)", "18_lag_obesity")

**Interpretation of the lag plot (adult obesity):**
* At **lag 1** the points form a smooth curve very close to the diagonal, which means that obesity in one year is very similar to that of the previous year. Each lag is very similar to the one before it, showing a very strong **positive autocorrelation**.
* At **every** lag, all the points are **above** the diagonal, and the curve moves further and further away, showing that the prevalence of obesity in adults has increased over time: a strong **positive trend**.
* **The 9 panels resemble each other** and none stands out, so **there is no seasonality and there are no cycles**. There are no isolated points, so we can say that **there are no atypical years**.

**Conclusion:** this lag plot confirms that obesity in adults **has increased every year** since 1990 (from 13.96 % to 26.2 %, almost double). This may be due to changes in diet and lifestyle (more sedentary behaviour). The fact that the increase and its speed are slowing down may indicate that health campaigns are starting to have an effect, but they have not yet managed to reverse it. It is the risk factor that **got worse**. 

### 4.4 Safely managed drinking water (% of population)

In [ ]:
# wa: % of the population using safely managed drinking water, one row per year
wa = pd.read_csv("Safely managed drinking water​.csv")
wa = wa.loc[(wa["GEO_NAME_SHORT"] == "Canada") & (wa["DIM_DEG_URB"] == "TOTAL")]   # Canada, urban + rural together
wa = wa[["DIM_TIME", "PERCENT_POP_N"]]                                             # keep the year and the value
wa = wa.rename(columns={"DIM_TIME": "year", "PERCENT_POP_N": "safe_water_pct"})
wa = wa.sort_values("year").set_index("year")                                      # sort by year and set it as the index
wa

In [ ]:
# Time plot of safely managed drinking water
time_plot(wa["safe_water_pct"], "Population using safely managed drinking water, Canada 2000–2024", "% of population",
          "20_water")

**Interpretation:** the percentage of the population using safely managed drinking water stayed **flat at approximately 99.05 % from 2000 to 2004**, then **falls in a straight line** to **96.9 % in 2023**, and in 2024 it stays the same.

In [ ]:
# Lag plots of safely managed drinking water (lags 1 to 9)
lag_plot(wa["safe_water_pct"], "Lag plots: safely managed drinking water (%)", "21_lag_water")

**Interpretation of the lag plot (safe drinking water):**
* At **lags 1 and 2** the points are practically on the diagonal, which means that the percentage in one year is almost the same as in the previous year. There is a very strong **positive autocorrelation** at every lag.
* **There is a negative trend.** As the lag increases, the points sit further below the diagonal.
* **No lag stands out**, so **there is no seasonality and there are no cycles**. There are also no points off the line, which tells us that **there are no atypical years**.

**Overall conclusion:** this lag plot confirms that almost all of Canada's population has safe drinking water, approximately 97 %. However, this coverage **has declined little by little** since 2005. One possible explanation is that the population grows faster than the water infrastructure, especially in rural and remote communities.


### 4.5 Tobacco use (people aged 15+, %)

In [ ]:
# tb: % of people aged 15+ who use tobacco (the years are not evenly spaced)
tb = pd.read_csv("Tobacco use.csv")
tb = tb.loc[(tb["GEO_NAME_SHORT"] == "Canada") & (tb["DIM_SEX"] == "TOTAL")]   # Canada, both sexes
tb = tb[["DIM_TIME", "PERCENT_POP_N"]]                                         # keep the year and the value
tb = tb.rename(columns={"DIM_TIME": "year", "PERCENT_POP_N": "tobacco_pct"})
tb = tb.sort_values("year").set_index("year")                                  # sort by year and set it as the index

tb_obs = tb.loc[:2024]           # observed estimates
tb_proj = tb.loc[2024:]          # WHO projections (2025, 2030); starts at 2024 so the dashed line connects
print("Years with data:", list(tb.index))
tb

In [ ]:
fig = go.Figure()
# solid line: observed estimates (2000–2024)
fig.add_scatter(x=tb_obs.index, y=tb_obs["tobacco_pct"], mode="lines+markers", name="Estimates",
                line=dict(color="#2a78d6", width=2.5))
# dashed line: WHO projection (2024–2030)
fig.add_scatter(x=tb_proj.index, y=tb_proj["tobacco_pct"], mode="lines+markers", name="WHO projection",
                line=dict(color="#2a78d6", width=2, dash="dash"))
# grey band that marks the projection period
fig.add_vrect(x0=2024, x1=2030, fillcolor="lightgray", opacity=0.3, line_width=0)
fig.update_layout(title="Tobacco use (age 15+), Canada 2000–2030", xaxis_title="Year", yaxis_title="% of people aged 15+")
show(fig, "23_tobacco")

**Interpretation:** Tobacco use in Canada among people aged 15 and older has been decreasing, from **28.9 % (2000) to 11.2 % (2024)**. It fell faster in the 2000s and more slowly in recent years. The WHO projects that it will keep falling: **10.8 % in 2025 and 8.9 % in 2030**.
**Lag plots cannot be made** because the data **are not equally spaced in time**: between 2000 and 2005 there are 5 years,
between 2005 and 2007 there are 2, and since 2022 there is 1.

**Conclusion:** tobacco has a **strong and steady negative trend**. With few data points at irregular intervals, seasonality cannot be assessed, but the series shows no increases and no atypical years.

### 4.6 Health statistics: summary

| Indicator | Period | Change | Trend | Seasonality | Outliers |
|---|---|---|---|---|---|
| Hypertension (30–79) | 1990–2019 | 33.0 % - 22.1 % | Negative (steady) | No | No |
| Alcohol (L per capita 15+) | 2000–2022 | 9.4 -10.3 (2008)  10.0 L | Up, then stable | No | No |
| Adult obesity (18+) | 1990–2022 | 14.0 % - 26.2 % | Positive (slowing) | No | No |
| Safe drinking water | 2000–2024 | 99.0 % - 96.9 % | Slightly negative | No | No |
| Tobacco use (15+) | 2000–2024 | 28.9 % - 11.2 % | Negative (strong) | Not testable | - |



In [ ]:
# panels: one entry per indicator of Part 4
# (panel title, series, where to write the first and the last value)
panels = [
    ("Hypertension (age 30–79), %", hy["hypertension_rate"], ["top center", "top center"]),
    ("Alcohol (litres per capita, age 15+)", al["alcohol_litres"], ["bottom right", "bottom center"]),
    ("Adult obesity (age 18+), %", ob["obesity_rate"], ["bottom right", "top center"]),
    ("Safely managed drinking water, %", wa["safe_water_pct"], ["top center", "top center"]),
    ("Tobacco use (age 15+), %", tb_obs["tobacco_pct"], ["top center", "top center"]),
]

# 2x3 grid, one panel per indicator (the sixth panel stays empty)
fig = make_subplots(rows=2, cols=3, subplot_titles=[title for title, s, pos in panels],
                    horizontal_spacing=0.07, vertical_spacing=0.18)
for i, (title, s, pos) in enumerate(panels):
    row, col = i // 3 + 1, i % 3 + 1   # position of this panel in the grid
    # line of the series
    fig.add_scatter(x=s.index, y=s, mode="lines", line=dict(color="#2a78d6", width=2.5), row=row, col=col)
    # first and last value of each series, written next to the point
    fig.add_scatter(x=[s.index[0], s.index[-1]], y=[s.iloc[0], s.iloc[-1]], mode="markers+text",
                    marker=dict(color="#2a78d6", size=8), text=[f"{s.iloc[0]:.1f}", f"{s.iloc[-1]:.1f}"],
                    textposition=pos, textfont=dict(color="black", size=12), cliponaxis=False, row=row, col=col)
    # pad: extra space above and below (15 % of the range) so the labels are not cut off
    pad = (s.max() - s.min()) * 0.15
    fig.update_yaxes(range=[s.min() - pad, s.max() + pad], row=row, col=col)

# alcohol: mark the maximum (2008)
fig.add_scatter(x=[al["alcohol_litres"].idxmax()], y=[al["alcohol_litres"].max()], mode="markers+text",
                marker=dict(color="#2a78d6", size=8), text=[f"{al['alcohol_litres'].max():.1f} (2008)"],
                textposition="top center", textfont=dict(color="black", size=12), row=1, col=2)

# tobacco: WHO projection (2025, 2030) as a dashed line
fig.add_scatter(x=tb_proj.index, y=tb_proj["tobacco_pct"], mode="lines", line=dict(color="#2a78d6", width=2, dash="dash"),
                row=2, col=2)
fig.add_annotation(x=2030, y=tb_proj["tobacco_pct"].iloc[-1], text=f"{tb_proj['tobacco_pct'].iloc[-1]:.1f} (WHO projection)",
                   showarrow=False, xanchor="right", yanchor="top", font=dict(size=11, color="gray"), row=2, col=2)
fig.update_yaxes(range=[6, 32], row=2, col=2)   # fixed Y range for the tobacco panel so the projection fits

fig.update_layout(title="Health statistics, Canada: risk factors and environment", showlegend=False)
show(fig, "25_health_statistics_summary", width=1200, height=700)